# Systematics fitting

In this notebook, we fit the instrumental systematics that will be later corrected for at the light curve stage.

In [ ]:
# Import necessary packages

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.colors as plc
import astropy.units as u
import astropy.constants as c
from astropy.io import fits
from astropy.time import Time
from scipy.optimize import minimize
from sunburn import analysis, tools, hst_observation, spectroscopy, systematics
import os

%config InlineBackend.figure_format = 'retina'

pylab.rcParams['figure.figsize'] = 9.0,6.5
pylab.rcParams['font.size'] = 18

For this exercise, we consider that each HST orbit is a "visit" and each subexposure is an "orbit". We do things this way just for the systematics fitting procedure. Ignore warnings that appear after the next cell.

## Visit A

In [ ]:
datasets_orbit1 = ['oe8ta1010_%i' % i for i in range(10)]
datasets_orbit2 = ['oe8ta1020_%i' % i for i in range(10)]
datasets_orbit3 = ['oe8ta1030_%i' % i for i in range(10)]
datasets_orbit4 = ['oe8ta1040_%i' % i for i in range(10)]
datasets_orbit5 = ['oe8ta1050_%i' % i for i in range(10)]

visit1 = hst_observation.Visit(datasets_orbit1, 'stis', prefix='../data/v1/splits10/') # Modify prefix accordingly, should point to the x1d files directory
visit2 = hst_observation.Visit(datasets_orbit2, 'stis', prefix='../data/v1/splits10/') # Modify prefix accordingly, should point to the x1d files directory
visit3 = hst_observation.Visit(datasets_orbit3, 'stis', prefix='../data/v1/splits10/') # Modify prefix accordingly, should point to the x1d files directory
visit4 = hst_observation.Visit(datasets_orbit4, 'stis', prefix='../data/v1/splits10/') # Modify prefix accordingly, should point to the x1d files directory
visit5 = hst_observation.Visit(datasets_orbit5, 'stis', prefix='../data/v1/splits10/') # Modify prefix accordingly, should point to the x1d files directory

/Users/ldsantos/Coding/sunburn/sunburn/hst_observation.py:1212: UserWarning: Could not find the jitter file (oe8ta1050_6_jit.fits).
  warn('Could not find the jitter file (%s_jit.fits).'
/Users/ldsantos/Coding/sunburn/sunburn/hst_observation.py:1212: UserWarning: Could not find the jitter file (oe8ta1050_7_jit.fits).
  warn('Could not find the jitter file (%s_jit.fits).'
/Users/ldsantos/Coding/sunburn/sunburn/hst_observation.py:1212: UserWarning: Could not find the jitter file (oe8ta1050_8_jit.fits).
  warn('Could not find the jitter file (%s_jit.fits).'
/Users/ldsantos/Coding/sunburn/sunburn/hst_observation.py:1212: UserWarning: Could not find the jitter file (oe8ta1050_9_jit.fits).
  warn('Could not find the jitter file (%s_jit.fits).'


In [ ]:
# HST ephemeris
name = 'HST'
period = 95.42 * u.min
reference_midpoint = Time(2459048.4408911085, format='jd')
duration = 0.5 * u.h

# The "transit" object -- we use it just to fold the light curves to HST's orbit
transit = analysis.Transit(planet_name=name, period=period,
                           transit_midpoint=reference_midpoint, duration14=duration)

line_list = spectroscopy.COSFUVLineList(wavelength_shift=.0, range_factor=1.0).lines

In [ ]:
# Wavelength range, excluding geocoronal contamination
wl_range = np.array([[1200, 1215.46744803], [1215.87295197, 1248]])

lc1 = analysis.LightCurve(visit1, transit, line_list)
lc1.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc2 = analysis.LightCurve(visit2, transit, line_list)
lc2.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc3 = analysis.LightCurve(visit3, transit, line_list)
lc3.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc4 = analysis.LightCurve(visit4, transit, line_list)
lc4.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc5 = analysis.LightCurve(visit5, transit, line_list)
lc5.compute_flux(wavelength_range=wl_range, poisson_regime=True)

alpha=1
mk_s = 10

lc1.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C0', transit_lines=False)
lc2.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C1', transit_lines=False)
lc3.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C2', transit_lines=False)
lc4.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C3', transit_lines=False)
lc5.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C4', transit_lines=False)

plt.tight_layout()
plt.show()

In [ ]:
scale = 1E-13

lc_list = [lc1, lc2, lc3, lc4, lc5]
phase_all = np.reshape((np.array([lc.phase for lc in lc_list]) * u.h).to(u.min).value / 95.42, 50)
flux_all = np.reshape(np.array([lc.integrated_flux for lc in lc_list]) / scale, 50)
unc_all = np.reshape(np.array([lc.f_unc for lc in lc_list]) / scale, 50)

In [ ]:
# The systematics model
def breathing_model(phase_hst_k, a_i, n):
    if n > 0:
        return 1 + np.sum(a_i * np.sin(2 * np.pi * np.arange(1, n + 1) * phase_hst_k))
    else:
        return 1

def visit_systematic_model(phase_hst, k_orb, a_i, n):
    n_orbits = len(k_orb)
    systematic = []
    for i in range(n_orbits):
        systematic.append(k_orb[i] * np.array([
            breathing_model(pk, a_i, n) for pk in phase_hst[i]]))
    return np.array(systematic)

In [ ]:
# Let's fit the systematic model!

def log_likelihood(theta, x, y, yerr, n_i):
    
    # Figure out some important stuff
    n_fit_params = len(theta)
    n_orbits = n_fit_params - n_i
    n_datapoints_per_orbit = len(x) // n_orbits  # Assuming all orbits have the same number of subexposures
    k_orb = theta[:n_orbits]
    a_i = np.array(theta[n_orbits:])
    phase = np.reshape(np.array(x), (n_orbits, n_datapoints_per_orbit))
    flux = np.reshape(np.array(y), (n_orbits, n_datapoints_per_orbit))
    f_unc = np.reshape(np.array(yerr), (n_orbits, n_datapoints_per_orbit))
    
    model = np.zeros((n_orbits, n_datapoints_per_orbit))
    for k in range(n_orbits):
        model[k, :] = k_orb[k] * np.array([breathing_model(pk, a_i, n_i) for pk in phase[k]])
    sigma2 = f_unc ** 2
    return -0.5 * np.sum((flux - model) ** 2 / sigma2 + np.log(sigma2))

In [ ]:
nll = lambda *args: -log_likelihood(*args)
n_i = 1
guess = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 0.9,
                  ])
soln = minimize(nll, guess, args=(phase_all, flux_all, unc_all, n_i), method='TNC', options={'maxiter': 10000})
print(soln)
bic = len(guess) * np.log(len(phase_all)) - 2 * log_likelihood(soln.x, phase_all, flux_all, unc_all, n_i)
print(bic)

In [ ]:
pylab.rcParams['figure.figsize'] = 9.0,4.5
pylab.rcParams['font.size'] = 18

ax = plt.subplot()

phase_plot = (np.array([lc.phase for lc in lc_list]) * u.h).to(u.min).value / 95.42
flux_plot = np.array([lc.integrated_flux for lc in lc_list]) / scale
unc_plot = np.array([lc.f_unc for lc in lc_list]) / scale

# Parse the solution
n_orbits = np.shape(phase_plot)[0]
a_solution = np.array(soln.x[n_orbits:])
k_solution = np.array(soln.x[:n_orbits])
print(phase_plot)
for i in range(n_orbits):
    high_res_phase = np.linspace(-0.20, 0.20, 100)
    best_solution = k_solution[i] * np.array([breathing_model(pk, a_solution, n_i) for pk in high_res_phase])
    ax.plot(high_res_phase, best_solution, color='C%i' % i, ls='--')
    ax.errorbar(phase_plot[i], flux_plot[i], yerr=unc_plot[i], fmt='o', color='C%i' % i, label='Orbit %i' % (i + 1))

ax.set(xlabel='Phase', ylabel=r'Flux ($\times 10^{-13}$ erg s$^{-1}$ cm$^{-2}$)',
       xlim=(-0.20, 0.20))
plt.legend(fontsize=11)
plt.tight_layout()
# plt.savefig('systematics_v1.pdf')
plt.show()

In [ ]:
# Correction factors
corr_factor = []
for i in range(n_orbits):
    best_solution = np.array([breathing_model(pk, a_solution, n_i) for pk in phase_plot[i]])
    corr_factor.append(1 / best_solution)
corr_factor = np.array(corr_factor)
np.savetxt('10splits_breathing_corr_v1.txt', corr_factor)
print(corr_factor)

## Visit C

In [ ]:
datasets_orbit1 = ['oen501010_%i' % i for i in range(10)]
datasets_orbit2 = ['oen501020_%i' % i for i in range(10)]
datasets_orbit3 = ['oen501030_%i' % i for i in range(10)]
datasets_orbit4 = ['oen501040_%i' % i for i in range(10)]
datasets_orbit5 = ['oen501050_%i' % i for i in range(10)]

visit1 = hst_observation.Visit(datasets_orbit1, 'stis', prefix='../data/v3/splits10/')
visit2 = hst_observation.Visit(datasets_orbit2, 'stis', prefix='../data/v3/splits10/')
visit3 = hst_observation.Visit(datasets_orbit3, 'stis', prefix='../data/v3/splits10/')
visit4 = hst_observation.Visit(datasets_orbit4, 'stis', prefix='../data/v3/splits10/')
visit5 = hst_observation.Visit(datasets_orbit5, 'stis', prefix='../data/v3/splits10/')

/Users/ldsantos/Coding/sunburn/sunburn/hst_observation.py:1212: UserWarning: Could not find the jitter file (oen501050_7_jit.fits).
  warn('Could not find the jitter file (%s_jit.fits).'
/Users/ldsantos/Coding/sunburn/sunburn/hst_observation.py:1212: UserWarning: Could not find the jitter file (oen501050_8_jit.fits).
  warn('Could not find the jitter file (%s_jit.fits).'
/Users/ldsantos/Coding/sunburn/sunburn/hst_observation.py:1212: UserWarning: Could not find the jitter file (oen501050_9_jit.fits).
  warn('Could not find the jitter file (%s_jit.fits).'


In [ ]:
name = 'HST'
period = 95.42 * u.min
reference_midpoint = Time((2459805.5001238114 + 2459805.5023855134) / 2, format='jd')
duration = 0.5 * u.h

# The "transit" object -- we use it just to fold the light curves to HST's orbit
transit = analysis.Transit(planet_name=name, period=period,
                           transit_midpoint=reference_midpoint, duration14=duration)

line_list = spectroscopy.COSFUVLineList(wavelength_shift=.0, range_factor=1.0).lines

In [ ]:
wl_range = np.array([[1200, 1215.46744803], [1215.87295197, 1248]])

lc1 = analysis.LightCurve(visit1, transit, line_list)
lc1.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc2 = analysis.LightCurve(visit2, transit, line_list)
lc2.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc3 = analysis.LightCurve(visit3, transit, line_list)
lc3.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc4 = analysis.LightCurve(visit4, transit, line_list)
lc4.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc5 = analysis.LightCurve(visit5, transit, line_list)
lc5.compute_flux(wavelength_range=wl_range, poisson_regime=True)

alpha=1
mk_s = 10

lc1.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C0', transit_lines=False)
lc2.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C1', transit_lines=False)
lc3.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C2', transit_lines=False)
lc4.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C3', transit_lines=False)
lc5.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C4', transit_lines=False)
plt.tight_layout()
plt.show()

In [ ]:
scale = 1E-13

lc_list = [lc1, lc2, lc3, lc4, lc5]
phase_all = np.reshape((np.array([lc.phase for lc in lc_list]) * u.h).to(u.min).value / 95.42, 50)
flux_all = np.reshape(np.array([lc.integrated_flux for lc in lc_list]) / scale, 50)
unc_all = np.reshape(np.array([lc.f_unc for lc in lc_list]) / scale, 50)

In [ ]:
nll = lambda *args: -log_likelihood(*args)
n_i = 1
guess = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 0.9,
                  ])
soln = minimize(nll, guess, args=(phase_all, flux_all, unc_all, n_i), method='TNC', options={'maxiter': 10000})
print(soln)
bic = len(guess) * np.log(len(phase_all)) - 2 * log_likelihood(soln.x, phase_all, flux_all, unc_all, n_i)
print(bic)

In [ ]:
pylab.rcParams['figure.figsize'] = 9.0,4.5
pylab.rcParams['font.size'] = 18

ax = plt.subplot()

phase_plot = (np.array([lc.phase for lc in lc_list]) * u.h).to(u.min).value / 95.42
flux_plot = np.array([lc.integrated_flux for lc in lc_list]) / scale
unc_plot = np.array([lc.f_unc for lc in lc_list]) / scale

# Parse the solution
n_orbits = len(phase_plot)
a_solution = np.array(soln.x[n_orbits:])#np.reshape(np.array(soln.x[n_orbits:]), (n_i, n_orbits)).T
k_solution = np.array(soln.x[:n_orbits])
for i in range(n_orbits):
    high_res_phase = np.linspace(-0.20, 0.20, 100)
    best_solution = k_solution[i] * np.array([breathing_model(pk, a_solution, n_i) for pk in high_res_phase])
    ax.plot(high_res_phase, best_solution, color='C%i' % i, ls='--')
    ax.errorbar(phase_plot[i], flux_plot[i], yerr=unc_plot[i], fmt='o', color='C%i' % i, label='Orbit %i' % (i + 1))

ax.set(xlabel='Phase', ylabel=r'Flux ($\times 10^{-13}$ erg s$^{-1}$ cm$^{-2}$)',
       xlim=(-0.20, 0.20))
# plt.legend(fontsize=11)
plt.tight_layout()
# plt.savefig('systematics_v3.pdf')
plt.show()

In [ ]:
# Correction factors
corr_factor = []
for i in range(n_orbits):
    best_solution = np.array([breathing_model(pk, a_solution, n_i) for pk in phase_plot[i]])
    corr_factor.append(1 / best_solution)
corr_factor = np.array(corr_factor)
np.savetxt('10splits_breathing_corr_v3.txt', corr_factor)
print(corr_factor)

## Visit D

In [ ]:
datasets_orbit1 = ['oen552010_%i' % i for i in range(10)]
datasets_orbit2 = ['oen552020_%i' % i for i in range(10)]
datasets_orbit3 = ['oen552030_%i' % i for i in range(10)]
datasets_orbit4 = ['oen552040_%i' % i for i in range(10)]
datasets_orbit5 = ['oen552050_%i' % i for i in range(10)]

visit1 = hst_observation.Visit(datasets_orbit1, 'stis', prefix='../data/v4/splits10/')
visit2 = hst_observation.Visit(datasets_orbit2, 'stis', prefix='../data/v4/splits10/')
visit3 = hst_observation.Visit(datasets_orbit3, 'stis', prefix='../data/v4/splits10/')
visit4 = hst_observation.Visit(datasets_orbit4, 'stis', prefix='../data/v4/splits10/')
visit5 = hst_observation.Visit(datasets_orbit5, 'stis', prefix='../data/v4/splits10/')

/Users/ldsantos/Coding/sunburn/sunburn/hst_observation.py:1214: UserWarning: Could not find the jitter file (oen552050_7_jit.fits).
  warn('Could not find the jitter file (%s_jit.fits).'
/Users/ldsantos/Coding/sunburn/sunburn/hst_observation.py:1214: UserWarning: Could not find the jitter file (oen552050_8_jit.fits).
  warn('Could not find the jitter file (%s_jit.fits).'
/Users/ldsantos/Coding/sunburn/sunburn/hst_observation.py:1214: UserWarning: Could not find the jitter file (oen552050_9_jit.fits).
  warn('Could not find the jitter file (%s_jit.fits).'


In [ ]:
name = 'HST'
period = 95.42 * u.min
transit_midpoint = Time((2460017.0066818837 + 2460017.0089434884) / 2, format='jd')
duration = 0.5 * u.h

# The "transit" object -- we use it just to fold the light curves to HST's orbit
transit = analysis.Transit(planet_name=name, period=period,
                           transit_midpoint=transit_midpoint, duration14=duration)

line_list = spectroscopy.COSFUVLineList(wavelength_shift=.0, range_factor=1.0).lines

In [ ]:
wl_range = np.array([[1200, 1215.46744803], [1215.87295197, 1248]])

lc1 = analysis.LightCurve(visit1, transit, line_list)
lc1.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc2 = analysis.LightCurve(visit2, transit, line_list)
lc2.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc3 = analysis.LightCurve(visit3, transit, line_list)
lc3.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc4 = analysis.LightCurve(visit4, transit, line_list)
lc4.compute_flux(wavelength_range=wl_range, poisson_regime=True)

lc5 = analysis.LightCurve(visit5, transit, line_list)
lc5.compute_flux(wavelength_range=wl_range, poisson_regime=True)

alpha=1
mk_s = 10

lc1.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C0', transit_lines=False)
lc2.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C1', transit_lines=False)
lc3.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C2', transit_lines=False)
lc4.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C3', transit_lines=False)
lc5.plot(fold=True, plot_splits=False, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=-15, symbol_color='C4', transit_lines=False)
plt.tight_layout()
plt.show()

In [ ]:
scale = 1E-13

lc_list = [lc1, lc2, lc3, lc4, lc5]
phase_all = np.reshape((np.array([lc.phase for lc in lc_list]) * u.h).to(u.min).value / 95.42, 50)
flux_all = np.reshape(np.array([lc.integrated_flux for lc in lc_list]) / scale, 50)
unc_all = np.reshape(np.array([lc.f_unc for lc in lc_list]) / scale, 50)

In [ ]:
nll = lambda *args: -log_likelihood(*args)
n_i = 1
guess = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 0.9
                  ])
soln = minimize(nll, guess, args=(phase_all, flux_all, unc_all, n_i), method='TNC', options={'maxiter': 10000})
print(soln)
bic = len(guess) * np.log(len(phase_all)) - 2 * log_likelihood(soln.x, phase_all, flux_all, unc_all, n_i)
print(bic)

In [ ]:
pylab.rcParams['figure.figsize'] = 9.0,4.5
pylab.rcParams['font.size'] = 18

ax = plt.subplot()

phase_plot = (np.array([lc.phase for lc in lc_list]) * u.h).to(u.min).value / 95.42
flux_plot = np.array([lc.integrated_flux for lc in lc_list]) / scale
unc_plot = np.array([lc.f_unc for lc in lc_list]) / scale

# Parse the solution
n_orbits = len(phase_plot)
a_solution = np.array(soln.x[n_orbits:])#np.reshape(np.array(soln.x[n_orbits:]), (n_i, n_orbits)).T
k_solution = np.array(soln.x[:n_orbits])
for i in range(n_orbits):
    high_res_phase = np.linspace(-0.20, 0.20, 100)
    best_solution = k_solution[i] * np.array([breathing_model(pk, a_solution, n_i) for pk in high_res_phase])
    ax.plot(high_res_phase, best_solution, color='C%i' % i, ls='--')
    ax.errorbar(phase_plot[i], flux_plot[i], yerr=unc_plot[i], fmt='o', color='C%i' % i, label='Orbit %i' % (i + 1))

ax.set(xlabel='Phase', ylabel=r'Flux ($\times 10^{-13}$ erg s$^{-1}$ cm$^{-2}$)',
       xlim=(-0.20, 0.20))
# plt.legend(fontsize=11)
plt.tight_layout()
# plt.savefig('systematics_v4.pdf')
plt.show()

In [ ]:
# Correction factors
corr_factor = []
for i in range(n_orbits):
    best_solution = np.array([breathing_model(pk, a_solution, n_i) for pk in phase_plot[i]])
    corr_factor.append(1 / best_solution)
corr_factor = np.array(corr_factor)
np.savetxt('10splits_breathing_corr_v4.txt', corr_factor)
print(corr_factor)